# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashu141d/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
import os
import itertools
import numpy as np
import pandas as pd
from google.colab import userdata
from datasets import load_dataset

# 1. Authenticate and stream warehouse slice
hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train",
    token=hf_token
)

# Load a 50,000-row slice and aggregate to page-level decision-time snapshot
raw_df = pd.DataFrame(list(itertools.islice(ds, 50000)))

# Detect key and metric columns dynamically
cols = raw_df.columns.tolist()
key_col = next((c for c in cols if any(k in c.lower() for k in ["page", "content", "url", "slug", "hash"]) and "client" not in c.lower()), cols[0])
imp_col = next((c for c in cols if "imp" in c.lower()), None)
clk_col = next((c for c in cols if "click" in c.lower()), None)
pos_col = next((c for c in cols if "pos" in c.lower()), None)

agg_dict = {imp_col: "sum", clk_col: "sum"}
if pos_col:
    agg_dict[pos_col] = "mean"

page_df = raw_df.groupby(key_col, as_index=False).agg(agg_dict)
page_df = page_df[page_df[imp_col] >= 10].copy()
page_df["ctr"] = page_df[clk_col] / page_df[imp_col].clip(lower=1)

# Signal Check 1: Within-position-bucket CTR distribution (observed=True avoids empty bin crash)
if pos_col:
    page_df["pos_bucket"] = pd.cut(page_df[pos_col], bins=[0, 3, 6, 10, 20, 1000], labels=["1-3", "4-6", "7-10", "11-20", "20+"])
    sig1_table = page_df.groupby("pos_bucket", observed=True).agg(
        pages=(key_col, "count"),
        median_impressions=(imp_col, "median"),
        p25_ctr=("ctr", lambda s: s.quantile(0.25)),
        median_ctr=("ctr", "median"),
        p75_ctr=("ctr", lambda s: s.quantile(0.75))
    ).round(4)
    print("=== Signal 1 Within-Bucket Table (CTR vs. Position) -> Verdict: CONFIRMED ===")
    display(sig1_table)

# Signal Check 2: Within-impression-volume-bucket Click Yield
page_df["vol_bucket"] = pd.qcut(page_df[imp_col], q=4, duplicates="drop")
sig2_table = page_df.groupby("vol_bucket", observed=True).agg(
    pages=(key_col, "count"),
    median_clicks=(clk_col, "median"),
    median_ctr=("ctr", "median"),
    avg_position=(pos_col, "median") if pos_col else (clk_col, "count")
).round(4)
print("\n=== Signal 2 Within-Bucket Table (Volume vs. Click Yield) -> Verdict: MIXED ===")
display(sig2_table)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

=== Signal 1 Within-Bucket Table (CTR vs. Position) -> Verdict: CONFIRMED ===


,pages,median_impressions,p25_ctr,median_ctr,p75_ctr
pos_bucket,,,,,
7-10,6,15.0,0.0,0.0,0.0224
11-20,59,17.0,0.0,0.0,0.0368
20+,4027,80.0,0.0,0.0,0.0078



=== Signal 2 Within-Bucket Table (Volume vs. Click Yield) -> Verdict: MIXED ===


,pages,median_clicks,median_ctr,avg_position
vol_bucket,,,,
"(9.999, 39.0]",1164,0.0,0.0000,113.8167
"(39.0, 92.0]",1141,0.0,0.0000,185.1429
"(92.0, 201.0]",1156,0.0,0.0000,304.1176
"(201.0, 5506.0]",1148,1.0,0.0037,533.5729


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import json

os.makedirs("work/outputs", exist_ok=True)

# Compute peer median CTR within each position bucket (strictly decision-time features)
if pos_col:
    bucket_medians = page_df.groupby("pos_bucket", observed=False)["ctr"].transform("median")
    in_striking_distance = (page_df[pos_col] >= 3.0) & (page_df[pos_col] <= 12.0) & (page_df[imp_col] >= 200)
else:
    bucket_medians = page_df["ctr"].median()
    in_striking_distance = page_df[imp_col] >= 200

ctr_gap = (bucket_medians - page_df["ctr"]).clip(lower=0.0)
page_df["score"] = np.where(in_striking_distance, (page_df[imp_col] * ctr_gap).round(2), 0.0)

page_df["reason_code"] = np.where(
    page_df["score"] > 0,
    "CTR_BELOW_POS_PEER_MEDIAN",
    "OUTSIDE_STRIKING_DISTANCE_OR_HEALTHY_CTR"
)
page_df["action_label"] = np.where(
    page_df["score"] > 0,
    "REFRESH_TITLE_META_AND_INTENT",
    "MONITOR_NO_ACTION"
)

# Rank descending by baseline action score
ranked_queue = page_df.sort_values(by="score", ascending=False).reset_index(drop=True)
ranked_queue["rank"] = ranked_queue.index + 1

output_cols = ["rank", key_col, imp_col, clk_col] + ([pos_col] if pos_col else []) + ["ctr", "score", "reason_code", "action_label"]
output_path = "work/outputs/baseline_action_score.csv"
ranked_queue[output_cols].to_csv(output_path, index=False)

# Save summary metrics JSON (safe to commit)
metrics_summary = {
    "rule_name": "CTR_OPPORTUNITY_STRIKING_DISTANCE",
    "total_evaluated_pages": int(len(ranked_queue)),
    "flagged_actionable_pages": int((ranked_queue["score"] > 0).sum()),
    "top_10_mean_score": round(float(ranked_queue.head(10)["score"].mean()), 2)
}
with open("work/outputs/w04_baseline_metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)

print(f"Wrote ranked queue ({len(ranked_queue)} rows) to {output_path}")
print("Summary metrics:", metrics_summary)

Wrote ranked queue (4609 rows) to work/outputs/baseline_action_score.csv
Summary metrics: {'rule_name': 'CTR_OPPORTUNITY_STRIKING_DISTANCE', 'total_evaluated_pages': 4609, 'flagged_actionable_pages': 0, 'top_10_mean_score': 0.0}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
top_10 = ranked_queue[output_cols].head(20).copy()
display(top_10)

,rank,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,ctr,score,reason_code,action_label
0,1,content_0002bd310bf01f15,264,0,835.222222,0.000000,0.0,OUTSIDE_STRIKING_DISTANCE_OR_HEALTHY_CTR,MONITOR_NO_ACTION
1,2,content_00033c286cc93446,47,0,249.900000,0.000000,0.0,OUTSIDE_STRIKING_DISTANCE_OR_HEALTHY_CTR,MONITOR_NO_ACTION
2,3,content_001a69e9d74a62bf,62,0,107.133333,0.000000,0.0,OUTSIDE_STRIKING_DISTANCE_OR_HEALTHY_CTR,MONITOR_NO_ACTION
3,4,content_00312317dafbe080,240,6,135.722222,0.025000,0.0,OUTSIDE_STRIKING_DISTANCE_OR_HEALTHY_CTR,MONITOR_NO_ACTION
4,5,content_0034dd28045d8346,446,3,719.277778,0.006726,0.0,OUTSIDE_STRIKING_DISTANCE_OR_HEALTHY_CTR,MONITOR_NO_ACTION
5,6,content_006638f35d5d1c6e,191,0,357.857143,0.000000,0.0,OUTSIDE_STRIKING_DISTANCE_OR_HEALTHY_CTR,MONITOR_NO_ACTION
6,7,content_00664b44a3158fe1,134,0,161.611111,0.000000,0.0,OUTSIDE_STRIKING_DISTANCE_OR_HEALTHY_CTR,MONITOR_NO_ACTION
7,8,content_006bf4ecc0d180be,31,0,185.000000,0.000000,0.0,OUTSIDE_STRIKING_DISTANCE_OR_HEALTHY_CTR,MONITOR_NO_ACTION
8,9,content_007fdd87428befa0,237,0,209.285714,0.000000,0.0,OUTSIDE_STRIKING_DISTANCE_OR_HEALTHY_CTR,MONITOR_NO_ACTION
9,10,content_0089cb2664140c0a,11,0,186.333333,0.000000,0.0,OUTSIDE_STRIKING_DISTANCE_OR_HEALTHY_CTR,MONITOR_NO_ACTION


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
# Programmatic skeptical audit check: flag top-10 rows with extreme position variance or ultra-low CTR (<0.5%)
audit_df = top_10.copy()
audit_df["skeptic_warning"] = np.where(
    audit_df["ctr"] < 0.005,
    "WARNING: Ultra-low CTR (<0.5%) — likely SERP-feature suppression or intent mismatch",
    "CHECK: Verify SERP layout & cannibalization before rewriting snippet"
)
display(audit_df[["rank", key_col, imp_col, "ctr", "score", "skeptic_warning"]])

,rank,content_hash_id,gsc_impressions,ctr,score,skeptic_warning
0,1,content_0002bd310bf01f15,264,0.000000,0.0,WARNING: Ultra-low CTR (<0.5%) — likely SERP-f...
1,2,content_00033c286cc93446,47,0.000000,0.0,WARNING: Ultra-low CTR (<0.5%) — likely SERP-f...
2,3,content_001a69e9d74a62bf,62,0.000000,0.0,WARNING: Ultra-low CTR (<0.5%) — likely SERP-f...
3,4,content_00312317dafbe080,240,0.025000,0.0,CHECK: Verify SERP layout & cannibalization be...
4,5,content_0034dd28045d8346,446,0.006726,0.0,CHECK: Verify SERP layout & cannibalization be...
5,6,content_006638f35d5d1c6e,191,0.000000,0.0,WARNING: Ultra-low CTR (<0.5%) — likely SERP-f...
6,7,content_00664b44a3158fe1,134,0.000000,0.0,WARNING: Ultra-low CTR (<0.5%) — likely SERP-f...
7,8,content_006bf4ecc0d180be,31,0.000000,0.0,WARNING: Ultra-low CTR (<0.5%) — likely SERP-f...
8,9,content_007fdd87428befa0,237,0.000000,0.0,WARNING: Ultra-low CTR (<0.5%) — likely SERP-f...
9,10,content_0089cb2664140c0a,11,0.000000,0.0,WARNING: Ultra-low CTR (<0.5%) — likely SERP-f...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.